# 5 · Measuring how good an answer is
Turn *is this answer good?* into a number, so results can be compared before and after training.

| Type of answer | Measure | Example |
|---|---|---|
| one exact answer (a command, a number) | exact match | right or wrong: 1 or 0 |
| an answer with parts (a command) | part accuracy | 3 of 4 parts right → 0.75 |
| short text | word overlap | `red shirt` vs `the shirt is red` → 0.5 |
| based on documents (RAG) | faithfulness | 1 of 2 claims in the document → 0.5 |
| open text | a judge (a person or a model) | a score from 1 to 5 |

*No GPU needed. Part 5 (optional) uses a local Ollama model.*

## 1 · Exact match
1 if identical, 0 if not. Strict: even a small difference counts as wrong.

In [ ]:
def exact_match(answer, expected):
    return 1.0 if answer == expected else 0.0

print(exact_match('FIND shoes color=BK', 'FIND shoes color=BK'))
print(exact_match('find shoes color=BK', 'FIND shoes color=BK'))

## 2 · Part accuracy (for commands)
Partial credit: the share of the expected parts that appear in the answer.
The same idea as the score function in notebook 3.

In [ ]:
def part_accuracy(answer, expected):
    parts = expected.split()                        # e.g. FIND, shoes, color=BK, max=50
    found = [p for p in parts if p in answer.split()]
    return len(found) / len(parts)

expected = 'FIND shoes color=BK max=50'
print(part_accuracy('FIND shoes color=BK max=50', expected))
print(part_accuracy('FIND shoes color=black max=50', expected))
print(part_accuracy('black sneakers', expected))

## 3 · Word overlap (for short text)
The share of the expected words that appear in the answer.

In [ ]:
def word_overlap(answer, expected):
    words = expected.lower().split()
    found = [w for w in words if w in answer.lower().split()]
    return len(found) / len(words)

print(word_overlap('red shirt', 'the shirt is red'))
print(word_overlap('the shirt is red', 'the shirt is red'))
print(word_overlap('the shirt is not red', 'the shirt is red'))

**Keep in mind:** *the shirt is not red* gets a full 1.0 but means the opposite. Word overlap does not understand meaning; for that, a judge is needed (part 5).

## 4 · Faithfulness (for answers based on documents)
The answer should only say what the document says: supported claims ÷ all claims.
Tools such as RAGAS and DeepEval do this automatically with a judge model. Here it is done by hand.

In [ ]:
document = 'The shirt costs 20 dollars and is made of cotton.'
answer_claims = {
    'The shirt costs 20 dollars': True,      # in the document
    'The shirt comes in 3 colors': False,    # not in the document
}
print('faithfulness:', sum(answer_claims.values()) / len(answer_claims))

## 5 · A model as judge (optional)
For open answers, a second model gives a score from 1 to 5. Needs Ollama running with a local model.
A judge can be wrong: check some scores by hand.

In [ ]:
import requests

JUDGE = 'gemma4:e2b'                                # any local Ollama model

def judge(question, answer):
    prompt = (f'Question: {question}\nAnswer: {answer}\n\n'
              'Rate the answer from 1 (wrong) to 5 (perfect). Reply with only the number.')
    response = requests.post('http://localhost:11434/api/chat', timeout=120, json={
        'model': JUDGE,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
        'options': {'temperature': 0},
    })
    reply = response.json()['message']['content']
    digits = [ch for ch in reply if ch in '12345']
    return int(digits[0]) if digits else None        # None = no number in the reply

question = 'Why is the sky blue?'
for answer in ['Sunlight is scattered by the air, and blue light is scattered the most.',
               'Because the sky reflects the ocean.']:
    print(judge(question, answer), '  ', answer)

## Always
Test on examples never used for training, and compare before vs after with the same measure.